In [1]:
import sys
sys.path.insert(0, '../lib')

In [2]:
import collections
import functools
import joblib
import os
import pathlib

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.stats.multitest

import common_data

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200
%config InlineBackend.figure_format = "retina"

In [ ]:
BASE = common_data.DATA / '05_pseudobulk/58a_alive_vs_dead/alive_vs_dead'

In [ ]:
adata = sc.read_h5ad(common_data.SC_NORM)

We use all alive vs dead for gene expression sample filtering

In [6]:
sample_groups = pd.read_csv(BASE / '_group_labels.csv', index_col=0)

In [7]:
# cell type to set
genes_to_keep = {}

In [8]:
%%time
PSEUDOBULK_CELLS = 50
PSEUDOBULK_EXPR_IN_GROUP = 0.8
for ct in adata.obs.Level_6.unique():
    genes = None
    for group in sample_groups.group.unique():
        samples = sample_groups.bal_barcode[sample_groups.group.eq(group)]
        pseudobulks = []
        for sample in samples:
            idx = adata.obs.Level_6.eq(ct) & adata.obs.bal_barcode.eq(sample)
            if idx.sum() < PSEUDOBULK_CELLS:
                continue
            pseudobulks.append(adata.raw.X[idx, :].sum(axis=0).A1)
        if len(pseudobulks) == 0:
            continue
        pseudobulks = pd.DataFrame(pseudobulks, columns=adata.raw.var_names)
        expr_frac = (pseudobulks > 0).sum(axis=0) / pseudobulks.shape[0]
        group_genes = pseudobulks.columns[expr_frac.ge(PSEUDOBULK_EXPR_IN_GROUP)]
        if genes is None:
            genes = group_genes.to_numpy()
        else:
            genes = np.union1d(genes, group_genes)
    genes_to_keep[ct] = genes

CPU times: user 40.7 s, sys: 1.54 s, total: 42.3 s
Wall time: 42.4 s


In [9]:
{k: len(v) for k, v in genes_to_keep.items() if v is not None}

{'CD4 T cells': 10161,
 'CD8 T cells': 10417,
 'Mast cells': 9644,
 'NUPR1+ Macs': 12099,
 'B cells': 8783,
 'MRC1+C1QA+': 11515,
 'DC2': 10351,
 'MRC1+C1QA-': 11759,
 'Proliferating CD4 T cells': 10319,
 'Proliferating CD8 T cells': 10111,
 'Classical monocytes-2 IL1B': 9823,
 'Secretory cells': 12142,
 'Proliferating NUPR1+ Macs': 11330,
 'Tregs': 8751,
 'DC1': 9202,
 'Ionocytes': 13684,
 'Ciliated cells': 11696,
 'gdT cells': 8798,
 'Migratory DC': 8872,
 'Interstitial macrophages': 9414,
 'Hematopoietic stem cells': 13169,
 'Classical monocytes-1 CCR2': 8905,
 'pDC': 9865,
 'AT1 and AT2': 12478,
 'Proliferating plasma cells': 10737,
 'Non-classical monocytes': 9654,
 'Plasma cells': 9890,
 'Proliferating gdT cells': 9821}

In [10]:
genes_to_keep['Perivascular macrophages'] = genes_to_keep['Interstitial macrophages']

In [ ]:
PADJ_CUTOFF = 0.05
class ComparisonInfo:
    def __init__(self, control, condition, genes, genes_to_keep):
        self.control = control
        self.condition = condition
        self.genes_raw = genes
        self.filter_genes(genes_to_keep)

    def filter_genes(self, genes_to_keep):
        filtered_degs = self.genes_raw.loc[self.genes_raw.index.isin(genes_to_keep), :].copy()
        filtered_degs = filtered_degs.loc[filtered_degs.padj.notna()].copy()
        # recompute FDR correction on the filtered genes:
        filtered_degs['padj'] = statsmodels.stats.multitest.fdrcorrection(
            filtered_degs.pvalue,
            alpha=PADJ_CUTOFF
        )[1]
        # recompute gene status based on new `padj`
        filtered_degs['sign'] = ''
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.gt(0),
            'sign'
        ] = f'Up in {self.condition}'
        filtered_degs.loc[
            filtered_degs.padj.lt(PADJ_CUTOFF)
            & filtered_degs.log2FoldChange.lt(0),
            'sign'
        ] = f'Up in {self.control}'
        self.genes = filtered_degs


class CellTypeInfo:
    def __init__(self, path, task_info, genes_to_keep):
        self.path = path
        self.task_info = task_info
        self.comparisons = []
        self.meta = pd.read_csv(path / 'meta.csv', index_col=0)
        self.name = self.meta.cell_type.values[0]

        self.load_comparisons(genes_to_keep)

    def load_comparisons(self, genes_to_keep):
        for run in self.path.glob('**/degs.csv'):
            self.comparisons.append(
                ComparisonInfo(
                    self.task_info.column_values[0],
                    self.task_info.column_values[1],
                    pd.read_csv(run, index_col=0),
                    genes_to_keep[self.name]
                )
            )

    @property
    def n_comparisons(self):
        return len(self.comparisons)

In [12]:
class TaskData:
    def __init__(self, task, task_info):
        self.task = task
        self.task_info = task_info
        self.info = {}

In [14]:
%%time
data = {}
for task in BASE.parent.iterdir():
    task_info = common_data.TaskInfo(
        task.name,
        None,
        ['alive', 'dead'],
        None
    )
    task_data = TaskData(task, task_info)
    for cell_type_path in sorted((BASE.parent / task).iterdir()):
        if cell_type_path.name.startswith('.') or cell_type_path.name.startswith('_'):
            continue
        if not cell_type_path.is_dir():
            continue
        if not (cell_type_path / 'meta.csv').exists():
            continue
        info = CellTypeInfo(cell_type_path, task_info, genes_to_keep)
        if info.n_comparisons > 0:
            task_data.info[cell_type_path.name] = info
    if len(task_data.info) > 0:
        data[task.name] = task_data

CPU times: user 1.17 s, sys: 56.6 ms, total: 1.23 s
Wall time: 3.95 s


In [15]:
data

{'alive_vs_dead_viral': <__main__.TaskData at 0x145d12866260>,
 'alive_vs_dead_pathogen_negative': <__main__.TaskData at 0x145b2eab0ca0>,
 'alive_vs_dead': <__main__.TaskData at 0x145b2ddfa800>,
 'alive_vs_dead_bacterial': <__main__.TaskData at 0x145b2c16af50>}

In [ ]:
joblib.dump(data, '20c_deg_data.joblib')

In [ ]:
BASE2 = common_data.DATA / '05_pseudobulk/20c_degs'
for _, task in data.items():
    for k, ct_info in task.info.items():
        comp = ct_info.comparisons[0]
        deg_path = BASE2 / task.task_info.pathname / k / 'degs.csv'
        if deg_path.exists():
            print(f'File {deg_path} already exists, skipping')
            continue
        # Threshold GSEA analysis to at least 1000 genes in comparison
        if comp.genes.shape[0] < 1000:
            continue
        ct_path = BASE2 / task.task_info.pathname / k
        os.makedirs(ct_path, exist_ok=True)
        comp.genes.sort_values('log2FoldChange').to_csv(deg_path)